# Music Generation — Symbolic & Continuous
## CSE 253R Assignment 2
### Task 1: Symbolic Unconditioned Generation | Task 4: Continuous Conditioned Generation

---

## Overview

This notebook explores two music generation tasks:

- **Task 1 (Symbolic Unconditioned):** Train a model to learn the distribution
  p(Soprano, Alto, Tenor, Bass) on the **JSB Chorales** dataset and freely generate
  new 4-part Bach-style chorales.
- **Task 4 (Continuous Conditioned):** Fine-tune **MusicGen-small** (Meta/AudioCraft)
  on the **FMA-small** dataset to generate genre-specific audio from text prompts.

Both tasks share a common theme — learning to generate music — but differ in
representation (symbolic MIDI vs continuous audio) and conditioning (none vs text).


---
## Section 1: Dataset, Preprocessing & Exploratory Analysis

### 1.1 Dataset Context

The **JSB Chorales** (J.S. Bach, BWV 1–438) are a canonical benchmark in symbolic
music generation. Each chorale is a four-part SATB (Soprano, Alto, Tenor, Bass)
harmonization of a Lutheran hymn melody. Properties that make this ideal:

- **Small and clean:** ~370 four-part chorales, well-curated via `music21`
- **Homogeneous style:** All written by one composer under consistent rules
- **Clear structure:** Fixed 4-voice polyphony with well-defined voice ranges
- **Rich prior work:** DeepBach (Hadjeres 2017), BachBot (Liang 2017), BacHMMachine (Hahn 2021)
- **Clear evaluation:** Ground-truth harmonizations exist for Task 2

The chorales are loaded directly from `music21`'s built-in corpus — no external download needed.


In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ── Load preprocessed data ────────────────────────────────────────────────────
chorales    = np.load('jsb_chorales.npy', allow_pickle=True)
with open('split.json') as f:   split = json.load(f)
with open('vocab.json') as f:   vdata = json.load(f)

vocab      = vdata['vocab']
token2idx  = {int(k): v for k, v in vdata['token2idx'].items()}
VOCAB_SIZE = len(vocab)
VOICE_NAMES = ['Soprano', 'Alto', 'Tenor', 'Bass']
NOTE_NAMES  = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
VOICE_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

print(f"Dataset: {len(chorales)} chorales")
print(f"Split  : train={len(split['train'])}, val={len(split['val'])}, test={len(split['test'])}")
print(f"Vocab  : {VOCAB_SIZE} tokens (0=rest + {VOCAB_SIZE-1} unique MIDI pitches)")

### 1.2 Piano-roll Representation

In [ ]:
# Visualise a single chorale as a piano roll
fig, axes = plt.subplots(4, 1, figsize=(14, 6), sharex=True)
sample = chorales[42][:128]   # first 128 16th-notes of chorale 42

for vi, (ax, vname, color) in enumerate(zip(axes, VOICE_NAMES, VOICE_COLORS)):
    pitch_seq = sample[:, vi]
    times = np.arange(len(pitch_seq))
    # Draw sustain bars for each note
    t = 0
    while t < len(pitch_seq):
        p = pitch_seq[t]
        if p == 0:
            t += 1
            continue
        dur = 1
        while t + dur < len(pitch_seq) and pitch_seq[t+dur] == p:
            dur += 1
        ax.barh(p, dur, left=t, height=0.7, color=color, alpha=0.85)
        t += dur
    ax.set_ylabel(vname, fontsize=9)
    ax.set_ylim(35, 82)
    ax.yaxis.set_tick_params(labelsize=7)

axes[-1].set_xlabel('Time (16th notes)', fontsize=9)
fig.suptitle('Bach Chorale Piano Roll (BWV sample, first 32 beats)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_piano_roll.png', dpi=150, bbox_inches='tight')
plt.show()
print("Piano roll visualized.")

### 1.3 Voice Range Analysis

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for vi, (ax, vname, color) in enumerate(zip(axes, VOICE_NAMES, VOICE_COLORS)):
    pitches = np.concatenate([c[:, vi] for c in chorales])
    active  = pitches[pitches > 0]
    pc_hist = np.bincount(active % 12, minlength=12).astype(float)
    pc_hist /= pc_hist.sum()
    
    bars = ax.bar(range(12), pc_hist, color=color, alpha=0.8, edgecolor='white')
    ax.set_xticks(range(12))
    ax.set_xticklabels(NOTE_NAMES, fontsize=8, rotation=45)
    ax.set_title(f'{vname}\nMIDI {active.min()}–{active.max()}\n'
                 f'mean={active.mean():.1f}', fontsize=9)
    ax.set_ylabel('Frequency', fontsize=8)
    ax.set_ylim(0, 0.20)

fig.suptitle('Pitch Class Distribution by Voice', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_pitch_class.png', dpi=150, bbox_inches='tight')
plt.show()

# Print voice range table
print(f"{'Voice':<10} {'Min MIDI':<10} {'Max MIDI':<10} {'Mean':<8} {'Std':<8} {'Unique'}")
print("-" * 56)
for vi, vname in enumerate(VOICE_NAMES):
    pitches = np.concatenate([c[:, vi] for c in chorales])
    active  = pitches[pitches > 0]
    print(f"{vname:<10} {active.min():<10} {active.max():<10} "
          f"{active.mean():<8.1f} {active.std():<8.2f} {len(np.unique(active))}")

### 1.4 Chorale Length & Polyphony Distribution

In [ ]:
lengths = [len(c) / 4 for c in chorales]   # convert to beats

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Length distribution
axes[0].hist(lengths, bins=25, color='#2c3e50', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean={np.mean(lengths):.1f}')
axes[0].set_xlabel('Length (quarter note beats)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chorale Length Distribution')
axes[0].legend()

# Interval histogram (all voices)
all_intervals = []
for c in chorales:
    for vi in range(4):
        voice = c[:, vi]
        active = voice[voice > 0]
        if len(active) > 1:
            all_intervals.extend(np.diff(active.astype(int)).tolist())

iv_counts = Counter(all_intervals)
ivs_plot  = range(-12, 13)
iv_vals   = [iv_counts.get(i, 0) for i in ivs_plot]
colors_iv = ['#e74c3c' if i < 0 else '#2ecc71' if i > 0 else '#95a5a6' for i in ivs_plot]
axes[1].bar(ivs_plot, iv_vals, color=colors_iv, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Interval (semitones)')
axes[1].set_ylabel('Count')
axes[1].set_title('Melodic Interval Distribution\n(all voices, excl. unison)')
axes[1].set_xticks(range(-12, 13, 2))

# Vocabulary usage
all_pitches = np.concatenate([c.flatten() for c in chorales])
active_all  = all_pitches[all_pitches > 0]
pitch_counts = Counter(active_all.tolist())
pitches_sorted = sorted(pitch_counts.keys())
axes[2].bar(pitches_sorted,
            [pitch_counts[p] for p in pitches_sorted],
            color='#8e44ad', alpha=0.8, edgecolor='white', width=0.8)
axes[2].set_xlabel('MIDI Pitch')
axes[2].set_ylabel('Frequency')
axes[2].set_title(f'Pitch Usage Distribution\n(vocab size = {VOCAB_SIZE})')

fig.suptitle('JSB Chorales: Dataset Statistics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Length: min={min(lengths):.0f}, max={max(lengths):.0f}, "
      f"mean={np.mean(lengths):.1f} beats")
print(f"Intervals: {len(all_intervals):,} total")
print(f"  Steps (≤2 semitones): {sum(iv_counts[i] for i in range(-2,3))/len(all_intervals)*100:.1f}%")
print(f"  Leaps (>6 semitones): {sum(v for k,v in iv_counts.items() if abs(k)>6)/len(all_intervals)*100:.1f}%")

### 1.5 Preprocessing Pipeline

In [ ]:
# ── Complete preprocessing pipeline (shared by Task 1 and Task 2) ─────────────

class Tokenizer:
    """
    Maps MIDI pitch ↔ integer token.
    Token 0 = REST/PAD in both spaces.
    Vocabulary covers all 46 unique pitches found in JSB Chorales.
    """
    def __init__(self):
        self.vocab      = vocab          # list of MIDI values (0=rest)
        self.token2idx  = token2idx
        self.idx2token  = {v: int(k) for k, v in token2idx.items()}
        self.vocab_size = VOCAB_SIZE
        self.REST       = 0
    
    def encode(self, roll: np.ndarray) -> np.ndarray:
        """(T,4) MIDI pitch array → (T,4) integer token array."""
        out = np.zeros_like(roll)
        for t in range(roll.shape[0]):
            for v in range(4):
                out[t, v] = self.token2idx.get(int(roll[t, v]), self.REST)
        return out
    
    def decode(self, tokens: np.ndarray) -> np.ndarray:
        """(T,4) integer token array → (T,4) MIDI pitch array."""
        out = np.zeros_like(tokens)
        for t in range(tokens.shape[0]):
            for v in range(4):
                out[t, v] = self.idx2token.get(int(tokens[t, v]), 0)
        return out

def make_sequences(chorales, tokenizer, seq_len=64, stride=16, split_idx=None):
    """
    Slice chorales into overlapping windows.
    Returns X of shape (N, seq_len, 4) — integer token sequences.
    
    seq_len=64 = 16 quarter note beats = ~4 bars at typical tempo.
    stride=16  = slide by 4 beats between windows (75% overlap for train).
    """
    seqs  = []
    index = split_idx if split_idx is not None else range(len(chorales))
    for ci in index:
        enc = tokenizer.encode(chorales[ci])
        T   = len(enc)
        for start in range(0, T - seq_len, stride):
            seqs.append(enc[start:start + seq_len])
    return np.stack(seqs)

import pretty_midi

def roll_to_midi(roll: np.ndarray, out_path: str, bpm: int = 100):
    """
    Convert (T, 4) MIDI pitch array → .mid file.
    Each voice gets its own instrument track.
    """
    pm  = pretty_midi.PrettyMIDI(initial_tempo=float(bpm))
    sps = 60.0 / bpm / 4   # seconds per 16th-note step
    voice_programs = [0, 0, 0, 43]
    
    for vi in range(min(4, roll.shape[1])):
        inst = pretty_midi.Instrument(program=voice_programs[vi],
                                      name=VOICE_NAMES[vi])
        t = 0
        while t < len(roll):
            pitch = int(roll[t, vi])
            if pitch == 0:
                t += 1; continue
            dur = 1
            while t + dur < len(roll) and int(roll[t+dur, vi]) == pitch:
                dur += 1
            inst.notes.append(pretty_midi.Note(
                velocity=80, pitch=pitch,
                start=t * sps, end=(t + dur) * sps))
            t += dur
        pm.instruments.append(inst)
    pm.write(out_path)

# Verify pipeline
tokenizer = Tokenizer()
X_train = np.load('X_train.npy')
X_val   = np.load('X_val.npy')
X_test  = np.load('X_test.npy')

print("Preprocessing pipeline verified:")
print(f"  Tokenizer vocab_size : {tokenizer.vocab_size}")
print(f"  Sequences (train)    : {X_train.shape}  →  {X_train.shape[0]} windows of 64×4 tokens")
print(f"  Sequences (val)      : {X_val.shape}")
print(f"  Sequences (test)     : {X_test.shape}")

# Verify round-trip
sample_enc  = X_train[0]
sample_midi = tokenizer.decode(sample_enc)
sample_back = tokenizer.encode(sample_midi)
assert np.array_equal(sample_enc, sample_back), "Round-trip error!"
print(f"  Encode → Decode → Encode round-trip: ✓")

# Export a sample chorale to MIDI
roll_to_midi(chorales[0], 'sample_chorale_real.mid')
print(f"  Exported real chorale to sample_chorale_real.mid ✓")

---
## Section 2: Modeling — Task 1 (Unconditioned)

We implement two models for symbolic unconditioned generation:

1. **Bigram Markov Chain** — A simple baseline that learns per-voice transition probabilities
2. **LSTM** — A 2-layer LSTM with per-voice embeddings and output heads

Both models learn to predict the next timestep given the current one. The LSTM captures
longer-range dependencies through its hidden state, while the Markov chain is limited to
single-step context.


### 2.1 Bigram Markov Chain Baseline

For each voice independently, we build a 47×47 transition count matrix from training data,
apply Laplace (add-1) smoothing, and normalize to get conditional probabilities:

$$P(\text{next token} \mid \text{current token}) = \frac{\text{count}(\text{current} \to \text{next}) + 1}{\sum_{j} [\text{count}(\text{current} \to j) + 1]}$$

Generation proceeds autoregressively by sampling from the learned distribution.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from models import BigramMarkovChain, ChoraleLSTM, train_lstm, generate_lstm

# ── Fit Bigram Markov Chain ────────────────────────────────────────────────────
mc = BigramMarkovChain(vocab_size=VOCAB_SIZE)
mc.fit(X_train)

# Evaluate on test set
markov_test_ppl = mc.perplexity(X_test)
print(f"Bigram Markov Chain — Test Perplexity: {markov_test_ppl:.2f}")

# Visualize transition matrix for each voice
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for vi, (ax, vname) in enumerate(zip(axes, VOICE_NAMES)):
    im = ax.imshow(mc.transition_probs[vi], aspect='auto', cmap='hot')
    ax.set_title(f'{vname}', fontsize=10)
    ax.set_xlabel('Next token')
    if vi == 0:
        ax.set_ylabel('Current token')
fig.colorbar(im, ax=axes, shrink=0.6, label='P(next | current)')
fig.suptitle('Bigram Transition Probabilities (per voice)', fontweight='bold')
plt.tight_layout()
plt.savefig('markov_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

# Generate a sample chorale from the Markov chain
np.random.seed(123)
seed_token = X_train[0, 0, :]  # realistic seed from training data
markov_gen = mc.generate(length=128, temperature=0.8, seed_token=seed_token)
markov_gen_midi = tokenizer.decode(markov_gen)
roll_to_midi(markov_gen_midi, 'markov_chorale.mid')
print(f"\n✓ Generated Markov chorale (128 steps) → markov_chorale.mid")
print(f"  Token range: [{markov_gen.min()}, {markov_gen.max()}]")
print(f"  Unique tokens used: {len(np.unique(markov_gen))}")

### 2.2 LSTM Model Architecture

Our LSTM architecture processes all 4 voices jointly at each timestep:

```
Input: (batch, T, 4) token indices
  ↓
4 × Embedding(47, 64) → concatenate → (batch, T, 256)
  ↓
LSTM(256 input, 256 hidden, 2 layers, dropout=0.3)
  ↓
4 × Linear(256, 47) → logits: (batch, T, 4, 47)
```

**Teacher forcing:** During training, the model receives the ground-truth tokens at time t
and predicts tokens at time t+1. Loss is the sum of cross-entropy over all 4 voices.

**Generation:** Autoregressive sampling — start from a seed token, feed predictions back
as input for the next step. Temperature controls diversity.


In [ ]:
# ── Train LSTM ─────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training device: {device}")

model = ChoraleLSTM(
    vocab_size=VOCAB_SIZE,
    embed_dim=64,
    hidden_dim=256,
    n_layers=2,
    dropout=0.3,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# If pre-trained checkpoint exists, load it instead of re-training
import os
if os.path.exists('lstm_checkpoint.pt'):
    model.load_state_dict(torch.load('lstm_checkpoint.pt', map_location=device))
    model = model.to(device)
    print("\n✓ Loaded pre-trained checkpoint: lstm_checkpoint.pt")
    
    # Load saved history if available
    if os.path.exists('training_history.json'):
        with open('training_history.json') as f:
            history = json.load(f)
        print(f"  Epochs trained: {len(history['train_loss'])}")
        print(f"  Best val loss:  {min(history['val_loss']):.4f}")
        print(f"  Best val ppl:   {min(history['val_perplexity']):.2f}")
    else:
        history = None
else:
    history = train_lstm(
        model, X_train, X_val,
        epochs=50,
        batch_size=64,
        lr=1e-3,
        device=device,
    )
    torch.save(model.state_dict(), 'lstm_checkpoint.pt')
    with open('training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    print("\n✓ Model trained and saved.")

In [ ]:
# ── Plot Training Curves ───────────────────────────────────────────────────────
if history is not None:
    from models import plot_training_curves
    plot_training_curves(history)
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
else:
    print("No training history available (loaded from checkpoint).")

### 2.3 Perplexity Comparison: Markov vs LSTM


In [ ]:
# ── Test Perplexity ────────────────────────────────────────────────────────────
model.eval()
criterion = nn.CrossEntropyLoss(reduction='mean')

inp_test = torch.from_numpy(X_test[:, :-1, :]).long().to(device)
tgt_test = torch.from_numpy(X_test[:, 1:, :]).long().to(device)

with torch.no_grad():
    logits = model(inp_test)
    loss = 0.0
    for v in range(4):
        loss += criterion(
            logits[:, :, v, :].reshape(-1, VOCAB_SIZE),
            tgt_test[:, :, v].reshape(-1),
        ).item()
    lstm_test_ppl = float(np.exp(loss / 4.0))

print(f\"{'Model':<25} {'Test Perplexity':>15}\")
print(\"-\" * 42)
print(f\"{'Bigram Markov Chain':<25} {markov_test_ppl:>15.2f}\")
print(f\"{'LSTM (2-layer, h=256)':<25} {lstm_test_ppl:>15.2f}\")
print(f\"{'Improvement':<25} {markov_test_ppl / lstm_test_ppl:>15.1f}x\")

### 2.4 Generation & Comparison

We generate chorales from both models and compare:
- Token distribution (should approximate real Bach)
- Piano roll visualisation
- MIDI audio quality


In [ ]:
# ── Generate from LSTM ─────────────────────────────────────────────────────────
# Generate the main deliverable at temperature 0.9
seed = X_train[10, 0:1, :]
lstm_gen = generate_lstm(
    model, tokenizer.decode, length=192, temperature=0.9,
    seed=seed, device=device
)
roll_to_midi(lstm_gen, 'symbolic_unconditioned.mid')
print("★ symbolic_unconditioned.mid exported (192 steps = ~48 beats)")

# Also generate at different temperatures for comparison
for temp in [0.7, 1.0, 1.2]:
    gen = generate_lstm(
        model, tokenizer.decode, length=128, temperature=temp,
        seed=X_train[42, 0:1, :], device=device
    )
    roll_to_midi(gen, f'lstm_chorale_t{temp:.1f}.mid')
    print(f"  ✓ lstm_chorale_t{temp:.1f}.mid exported")

In [ ]:
# ── Distribution Comparison ────────────────────────────────────────────────────
# Generate 20 samples from each model for statistical comparison
markov_samples = []
lstm_samples = []
for i in range(20):
    seed = X_train[i, 0:1, :]
    markov_samples.append(mc.generate(64, temperature=0.8, seed_token=seed[0]))
    lstm_samples.append(
        generate_lstm(model, None, 64, temperature=0.9, seed=seed, device=device)
    )

markov_all = np.stack(markov_samples)
lstm_all = np.stack(lstm_samples)

# Plot token distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
bins = np.arange(VOCAB_SIZE + 1) - 0.5

for ax, data, title, color in zip(
    axes,
    [X_test, markov_all, lstm_all],
    ['Real Bach (test)', 'Bigram Markov', 'LSTM (ours)'],
    ['#2c3e50', '#e74c3c', '#2ecc71'],
):
    ax.hist(data.flatten(), bins=bins, density=True, alpha=0.75,
            edgecolor='white', color=color, linewidth=0.5)
    ax.set_xlabel('Token index')
    ax.set_ylabel('Density')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(-0.5, VOCAB_SIZE - 0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Token Distribution: Real vs Generated', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Piano roll comparison
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
titles = ['Real Bach', 'Bigram Markov', 'LSTM']
rolls = [
    tokenizer.decode(X_test[0]),
    tokenizer.decode(markov_all[0]),
    tokenizer.decode(lstm_all[0]),
]

for ax, roll, title in zip(axes, rolls, titles):
    for vi, (vname, color) in enumerate(zip(VOICE_NAMES, VOICE_COLORS)):
        voice = roll[:, vi]
        t = 0
        while t < len(voice):
            p = voice[t]
            if p == 0:
                t += 1; continue
            dur = 1
            while t + dur < len(voice) and voice[t + dur] == p:
                dur += 1
            ax.barh(p, dur, left=t, height=0.7, color=color, alpha=0.7)
            t += dur
    ax.set_ylabel('MIDI Pitch')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(35, 82)

axes[-1].set_xlabel('Time (16th notes)')
fig.suptitle('Piano Roll Comparison: Real vs Generated', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('piano_roll_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ All comparisons generated.")

---
## Section 3: Dataset, Preprocessing & EDA — Task 4 (Continuous Conditioned: MusicGen Fine-tuning)

### 3.1 Dataset Context: FMA-Small

The **Free Music Archive (FMA)** is an open, Creative Commons-licensed audio dataset.
We use **FMA-small**: 8,000 tracks × 30 seconds, balanced across **8 genres**:
Hip-Hop, Pop, Folk, Experimental, Rock, International, Electronic, Instrumental.

Key properties that make this ideal for MusicGen fine-tuning:
- **Genre labels are clean and balanced** (1000 tracks per genre) — perfect text conditioning
- **30-second clips** match MusicGen's natural generation length
- **Available on HuggingFace** as `rpmon/fma-genre-classification` — no manual download
- **Creative Commons licensed** — legally usable for research
- **Prior work:** Used extensively in MIR classification; MusicGen genre fine-tuning directly
  mirrors recent work (IJCRT 2025)

### 3.2 Task Formulation

**Input:** Text prompt describing genre and style (e.g., `"upbeat electronic music with synths"`)  
**Output:** 30-second audio clip at 32kHz that matches the prompt style

We fine-tune **MusicGen-small** (300M parameters, Meta/AudioCraft) on FMA genre subsets.
Fine-tuning updates model weights on our data — satisfying the "train your own weights"
requirement while leveraging a powerful pretrained audio backbone.

This is a **Continuous Conditioned** task: the conditioning signal is text, and the output is
raw audio waveform (continuous), not symbolic notation.


In [ ]:
# ── Task 4: FMA Dataset EDA ───────────────────────────────────────────────────
# Note: Full audio loading requires the FMA dataset to be downloaded.
# This cell shows the structure and statistics from FMA metadata.
# On your training machine: pip install datasets && run the loader below.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# FMA-small genre distribution (from official dataset statistics)
GENRES = ['Hip-Hop', 'Pop', 'Folk', 'Experimental',
          'Rock', 'International', 'Electronic', 'Instrumental']
TRACKS_PER_GENRE = 1000   # perfectly balanced
GENRE_COLORS = plt.cm.Set2(np.linspace(0, 1, 8))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Genre distribution
axes[0].bar(range(8), [TRACKS_PER_GENRE]*8, color=GENRE_COLORS, edgecolor='white')
axes[0].set_xticks(range(8))
axes[0].set_xticklabels(GENRES, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Number of Tracks')
axes[0].set_title('FMA-Small: Genre Distribution\n(perfectly balanced)', fontweight='bold')
axes[0].set_ylim(0, 1200)
for i, v in enumerate([TRACKS_PER_GENRE]*8):
    axes[0].text(i, v+20, str(v), ha='center', fontsize=8)

# Dataset size breakdown
labels = ['Train\n(80%)', 'Validation\n(20%)']
sizes  = [6400, 1600]
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Train/Val Split\n(8000 total tracks)', fontweight='bold')

# Clip duration / sample rate info
specs = {
    'Duration (s)':     30,
    'Sample Rate (kHz)': 22.05,
    'Bit depth':        16,
    'Channels':         1,
}
y_pos = range(len(specs))
axes[2].barh(list(y_pos), list(specs.values()),
             color='#3498db', alpha=0.8, edgecolor='white')
axes[2].set_yticks(list(y_pos))
axes[2].set_yticklabels(list(specs.keys()), fontsize=9)
axes[2].set_title('Audio Clip Specifications\n(per track)', fontweight='bold')
for i, v in enumerate(specs.values()):
    axes[2].text(v + 0.2, i, str(v), va='center', fontsize=9)

plt.suptitle('FMA-Small Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fma_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("FMA-Small Summary:")
print(f"  Total tracks    : 8,000 (8 genres × 1,000 tracks)")
print(f"  Clip duration   : 30 seconds each")
print(f"  Total audio     : {8000*30/3600:.1f} hours")
print(f"  Download size   : ~8 GB (fma_small.zip)")
print(f"  License         : Creative Commons")
print(f"  HuggingFace     : rpmon/fma-genre-classification")

### 3.3 Why Fine-tuning MusicGen vs Training from Scratch

In [ ]:
# Visualise the argument: fine-tuning vs. from scratch
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

data = [
    ['Criterion',          'Train from Scratch',      'Fine-tune MusicGen (Ours)'],
    ['Audio quality',      '❌ Noise/blur in 6 days', '✅ Studio-quality audio'],
    ['Dataset needed',     '100s of hours',            '~67 hrs (FMA-small)'],
    ['GPU time',           '~weeks',                   '~4–8 hours (Colab T4)'],
    ['Output variety',     '❌ Usually one mode',       '✅ 8 distinct genres'],
    ['Interesting output', '❌ Low probability',        '✅ High — real music'],
    ['Trains own weights', '✅ Yes',                    '✅ Yes (fine-tuning)'],
    ['Explainability',     'Low',                      'High — can ablate genres'],
]

table = ax.table(cellText=data[1:], colLabels=data[0],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.0)

# Style header
for j in range(3):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Style columns
for i in range(1, len(data)):
    table[i, 0].set_facecolor('#ecf0f1')
    table[i, 1].set_facecolor('#fadbd8')
    table[i, 2].set_facecolor('#d5f5e3')

plt.title('Modeling Choice: Fine-tuning MusicGen vs Training from Scratch',
          fontsize=12, fontweight='bold', pad=20)
plt.savefig('eda_fma_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 MusicGen Architecture Overview

In [ ]:
# MusicGen architecture summary diagram
fig, ax = plt.subplots(figsize=(13, 6))
ax.axis('off')

components = [
    (0.08, 0.5, 'Text Prompt\n"upbeat electronic\nwith synths"', '#3498db', 0.13),
    (0.30, 0.5, 'T5 Text\nEncoder\n(frozen)', '#9b59b6', 0.12),
    (0.52, 0.5, 'Transformer\nDecoder LM\n(300M params)\n★ fine-tuned', '#e74c3c', 0.14),
    (0.74, 0.5, 'EnCodec\nDecoder\n(frozen)', '#27ae60', 0.12),
    (0.92, 0.5, 'Audio\nOutput\n32kHz', '#f39c12', 0.10),
]

for x, y, label, color, w in components:
    box = mpatches.FancyBboxPatch((x - w/2, y - 0.18), w, 0.36,
                                   boxstyle='round,pad=0.02',
                                   facecolor=color, alpha=0.85,
                                   edgecolor='white', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=9, color='white', fontweight='bold')

# Arrows
for i in range(len(components) - 1):
    x1 = components[i][0]   + components[i][4]/2
    x2 = components[i+1][0] - components[i+1][4]/2
    ax.annotate('', xy=(x2, 0.5), xytext=(x1, 0.5),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))

# Fine-tuning annotation
ax.annotate('Only this component\nis fine-tuned on FMA',
            xy=(0.52, 0.32), xytext=(0.52, 0.10),
            ha='center', fontsize=9, color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('MusicGen Architecture: What We Fine-tune',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_musicgen_arch.png', dpi=150, bbox_inches='tight')
plt.show()

print("MusicGen-small key facts:")
print("  Parameters      : 300M (transformer decoder)")  
print("  Audio tokenizer : EnCodec (32kHz, 4 codebooks, frozen)")
print("  Text encoder    : T5-base (frozen)")
print("  Fine-tune target: Transformer decoder weights only")
print("  Training input  : (audio, genre_text_prompt) pairs")
print("  Install         : pip install audiocraft")
print("  HuggingFace     : facebook/musicgen-small")

### 3.5 Data Loading & Preprocessing Pipeline (Task 4)

In [ ]:
# ── Task 4 preprocessing pipeline ────────────────────────────────────────────
# This shows the full pipeline. Audio loading requires the FMA dataset.
# Run on your training machine with: pip install audiocraft datasets

def build_musicgen_dataset(genre: str, n_samples: int = 200,
                            audio_dir: str = 'fma_small/',
                            output_dir: str = 'musicgen_data/'):
    """
    Prepare a genre-specific subset of FMA-small for MusicGen fine-tuning.
    
    MusicGen's fine-tuning API expects:
      - Audio files (.mp3 or .wav), 30s clips
      - Matching .txt files with one text description per audio file
    
    Text prompts follow the template:
      "{genre} music" or more descriptive variants for diversity.
    
    Args:
        genre: One of the 8 FMA genres
        n_samples: Number of tracks to use (200 per genre = manageable fine-tune)
        audio_dir: Path to fma_small audio directory
        output_dir: Where to write (audio, description) pairs
    """
    import os, shutil
    os.makedirs(output_dir, exist_ok=True)
    
    # Genre-to-prompt mapping (richer than just genre name)
    PROMPTS = {
        'Hip-Hop':       'hip hop music with beats and rhythm',
        'Pop':           'upbeat pop music with melody',
        'Folk':          'acoustic folk music with guitar',
        'Experimental':  'experimental avant-garde music',
        'Rock':          'energetic rock music with electric guitar',
        'International': 'world music with traditional instruments',
        'Electronic':    'electronic music with synthesizers',
        'Instrumental':  'instrumental music without vocals',
    }
    
    prompt = PROMPTS.get(genre, f'{genre} music')
    # ... (load FMA audio files for this genre, write pairs)
    print(f"  Genre: {genre}")
    print(f"  Prompt: '{prompt}'")
    print(f"  Samples: {n_samples}")
    print(f"  Output: {output_dir}")

# Show genre→prompt mapping
print("Genre → Text Prompt Mapping (used for conditioning):")
print("-" * 55)
PROMPTS = {
    'Hip-Hop':       'hip hop music with beats and rhythm',
    'Pop':           'upbeat pop music with melody',
    'Folk':          'acoustic folk music with guitar',
    'Experimental':  'experimental avant-garde music',
    'Rock':          'energetic rock music with electric guitar',
    'International': 'world music with traditional instruments',
    'Electronic':    'electronic music with synthesizers',
    'Instrumental':  'instrumental music without vocals',
}
for genre, prompt in PROMPTS.items():
    print(f"  {genre:<15} → '{prompt}'")

print()
print("Fine-tuning strategy:")
print("  • Start from  : facebook/musicgen-small (pretrained)")
print("  • Fine-tune on: 200 tracks × 4 target genres (800 pairs)")
print("  • Epochs      : 20–30 (early stop on val loss)")
print("  • Batch size  : 4 (gradient accumulation ×4 = effective 16)")
print("  • Learning rate: 1e-4 (lower than pretraining to preserve features)")
print("  • Hardware    : ~4–6 hours on Colab T4 GPU")

---
## Section 4: Evaluation

We evaluate both tasks with quantitative metrics and qualitative listening.

**Task 1:** Perplexity, pitch-class KL divergence, interval histograms,
voice range violations, and parallel 5ths/octaves.

**Task 4:** Genre classifier consistency (pretrained vs fine-tuned MusicGen)
and side-by-side listening comparison.

Standalone scripts: `evaluate_task1.py`, `evaluate_task4.py`


### 4.1 Task 1 — Symbolic Evaluation


In [ ]:
# Run Task 1 evaluation (or load cached results)
import os, json
import subprocess

if not os.path.exists('evaluation_task1.json'):
    subprocess.run(['python', 'evaluate_task1.py', '--n-samples', '30'], check=True)

with open('evaluation_task1.json') as f:
    eval1 = json.load(f)

print('Task 1 Evaluation Results')
print('=' * 50)
print(f"{'Metric':<30} {'Markov':>10} {'LSTM':>10}")
print('-' * 50)
ppl = eval1['perplexity']
print(f"{'Test perplexity':<30} {ppl['markov']:>10.3f} {ppl['lstm']:>10.3f}")
kl = eval1['pitch_class_kl_to_real']
print(f"{'Pitch-class KL (→ Real)':<30} {kl['markov']:>10.4f} {kl['lstm']:>10.4f}")
l1 = eval1['interval_l1_to_real']
print(f"{'Interval L1 (→ Real)':<30} {l1['markov']:>10.4f} {l1['lstm']:>10.4f}")
vr = eval1['voice_range_violation_rate']
print(f"{'Voice range violations':<30} {vr['markov']:>10.4f} {vr['lstm']:>10.4f}")
pa = eval1['parallel_fifths_octaves_rate']
print(f"{'Parallel 5ths/octaves':<30} {pa['markov']:>10.4f} {pa['lstm']:>10.4f}")
print(f"\nReal Bach reference — range: {vr['real']:.4f}, parallel: {pa['real']:.4f}")


In [ ]:
from IPython.display import Image, display
import os

for img in ['eval_pitch_kl.png', 'eval_intervals.png', 'eval_summary.png']:
    if os.path.exists(img):
        print(img)
        display(Image(filename=img))
    else:
        print(f'  (run evaluate_task1.py to generate {img})')


**Task 1 Discussion:** The LSTM achieves lower perplexity than the bigram Markov
baseline, indicating better next-token prediction. Pitch-class KL and interval
distributions measure how Bach-like the generated chorales sound. Voice-leading
metrics check whether generated music respects SATB conventions.

Qualitative listening: compare `markov_chorale.mid`, `symbolic_unconditioned.mid`,
and `sample_chorale_real.mid`.


### 4.2 Task 4 — Continuous Conditioned Evaluation


In [ ]:
# Task 4 evaluation — genre consistency
import os, json
import subprocess

if not os.path.exists('evaluation_task4.json'):
    cmd = ['python', 'evaluate_task4.py']
    if not os.path.exists('musicgen_data'):
        cmd.append('--skip-classifier')
    subprocess.run(cmd, check=False)

if os.path.exists('evaluation_task4.json'):
    with open('evaluation_task4.json') as f:
        eval4 = json.load(f)

    print('Task 4 Evaluation — Genre Consistency')
    print('=' * 50)
    if 'pretrained' in eval4:
        print(f"  Pretrained MusicGen accuracy: {eval4['pretrained']['genre_accuracy']:.3f}")
    if 'finetuned' in eval4:
        print(f"  Fine-tuned MusicGen accuracy:  {eval4['finetuned']['genre_accuracy']:.3f}")
    if 'classifier_error' in eval4:
        print(f"  Note: {eval4['classifier_error']}")

    print('\nQualitative comparison (pretrained vs fine-tuned):')
    for row in eval4.get('comparison', []):
        print(f"  {row['genre']}: {row['prompt']}")
        print(f"    Pretrained: {row.get('pretrained_file', 'N/A')}")
        print(f"    Fine-tuned: {row.get('finetuned_file', 'N/A')}")
else:
    print('Run prepare_fma_data.py → musicgen_generate.py → evaluate_task4.py')


In [ ]:
from IPython.display import Image, display, Audio
import os
from pathlib import Path

if os.path.exists('eval_task4_genre_accuracy.png'):
    display(Image(filename='eval_task4_genre_accuracy.png'))

# Play generated samples if available
for d in ['generated_audio', 'generated_audio_finetuned']:
    p = Path(d)
    if p.exists():
        for f in sorted(p.glob('*.mp3'))[:2]:
            print(f'Listening: {f}')
            display(Audio(filename=str(f)))
        for f in sorted(p.glob('*.wav'))[:2]:
            print(f'Listening: {f}')
            display(Audio(filename=str(f)))

if os.path.exists('continuous_conditioned.mp3'):
    print('\n★ Main deliverable: continuous_conditioned.mp3')
    display(Audio(filename='continuous_conditioned.mp3'))


**Task 4 Discussion:** We fine-tune MusicGen-small on FMA genre subsets so that
text prompts like "hip hop music with beats" produce genre-consistent audio.
The genre classifier (trained on MFCC features) measures whether generated audio
matches the target genre better than the pretrained baseline.

Fine-tuning counts as training our own weights — we update the transformer decoder
on (audio, text) pairs from FMA-small.


---
## Section 5: Related Work

### Task 1 — Symbolic Unconditioned Generation

| Work | Model | Dataset | Key Contribution |
|---|---|---|---|
| Allan & Williams (2005) | HMM | Various | Classic baseline with BIC model selection |
| DeepBach (Hadjeres 2017) | Gibbs + LSTM | JSB Chorales | Steerable: any voice can be fixed |
| BachBot (Liang 2017) | LSTM seq2seq | JSB Chorales | Turing test fooled 1-in-3 listeners |
| Music Transformer (Huang 2019) | Transformer + relative attention | Piano-only | Best perplexity via relative positional encoding |
| BacHMMachine (Hahn 2021) | Theory-guided HMM | JSB Chorales | Interpretable chord transitions |

### Task 4 — Continuous Conditioned Generation

| Work | Model | Key Contribution |
|---|---|---|
| MusicGen (Copet 2023) | Single-stage AR LM over EnCodec | Efficient single-codebook interleaving |
| MusicLM (Agostinelli 2023) | Hierarchical audio LM | Semantic + acoustic token hierarchy |
| AudioCraft (Meta 2023) | Open-source framework | pip install audiocraft |
| FMA Dataset (Defferrard 2017) | - | 106K tracks, 161 genres, CC licensed |
| Genre Fine-tuning (IJCRT 2025) | MusicGen-small fine-tuned | Same FMA genre approach as ours |

Our work differs in combining both symbolic and continuous tasks,
providing a direct comparison of generation paradigms.
